# Distance-preserving graph control (experiment #1)

Closes the proximity confound in the headline mechanism result. The paper shows true edges (+0.046) >> random rewire (+0.012), but the random rewire's edges average **511 km** vs **92 km** for true edges, so a reviewer can blame distance, not topology. This control substitutes each true edge with the nearest-distance NON-upstream basin (in-degree preserved, proximity held near-fixed: distctrl 101 km vs true 92 km), destroying the topology while keeping proximity.

Pre-registration: `experiments/topology_ablation/preregistration_distance_control.md`. Predicted: distctrl falls to the random level (topology-specific). Falsified if distctrl ~= true (proximity-driven).

**Idempotent. Runtime -> T4 GPU -> Run all.** Skips any seed already complete.

## Cell 1 - Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 - Config (SEEDS = [11, 13, 17])

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[11,13,17]
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seeds', SEEDS)

## Cell 3 - Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 - Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 - Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 - GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 - Build the distance-preserving feature

Regenerates `features/upstream_q_distctrl_component0_lag1.p` from committed inputs (edge list + coords). Prints the validation: in-degree preserved, 0% overlap with true edges, distctrl ~101 km vs true ~92 km vs random ~511 km.

In [ ]:
%cd {REPO_DIR}
!python experiments/topology_ablation/build_distance_control.py --network component0 --lag-days 1 2>&1 | tail -12

## Cell 8 - Train L_upQdistctrl at seeds 11/13/17 (idempotent)

Stock cudalstm + the distance-control oracle feature, byte-identical config otherwise. Skips any seed whose test_metrics.csv already exists.

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
FEAT=f'{REPO_DIR}/experiments/topology_ablation/features'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
for SEED in SEEDS:
    if done('L_upQdistctrl',SEED):
        print(f'[skip] L_upQdistctrl seed{SEED} present'); continue
    print(f'\n########## L_upQdistctrl SEED {SEED} ##########')
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {SEED} --device cuda:0 --epochs 30 --feature-file {FEAT}/upstream_q_distctrl_component0_lag1.p --cond-name L_upQdistctrl 2>&1 | tail -6

## Cell 9 - Verdict: distctrl vs true vs random (connected basins, 3 seeds)

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle
from scipy.stats import wilcoxon
B=f'{REPO_DIR}/runs/topology_ablation/component0'
FEAT=f'{REPO_DIR}/experiments/topology_ablation/features'
feat=pickle.load(open(f'{FEAT}/upstream_q_distctrl_component0_lag1.p','rb'))
conn=[b for b in feat if np.abs(feat[b]['upstream_q'].values).mean()>0]
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
per_seed=[]; pooled=[]
for s in SEEDS:
    L=nse('L',s); X=nse('L_upQdistctrl',s)
    if L is None or X is None: print(f'seed {s}: missing run'); continue
    c=[b for b in conn if b in X.index and b in L.index]
    d=(X.loc[c]-L.loc[c]).values; per_seed.append(float(np.median(d))); pooled+=list(d)
pooled=np.array(pooled)
if len(pooled):
    p1=wilcoxon(pooled,alternative='greater').pvalue
    print(f'distctrl per-seed median Δ (connected) = '+' / '.join(f'{m:+.4f}' for m in per_seed))
    print(f'distctrl pooled median Δ = {np.median(pooled):+.4f}  (cross-seed mean {np.mean(per_seed):+.4f}), p={p1:.2e}, n={len(pooled)}')
    print('reference (MECHANISM_MULTISEED, oracle, connected): true +0.046 | reversed +0.031 | random +0.012')
    m=np.mean(per_seed)
    if m < 0.020: print('VERDICT: distctrl at/below the random level -> TOPOLOGY-SPECIFIC (proximity does not recover the gain). Pre-reg confirmed.')
    elif m > 0.038: print('VERDICT: distctrl ~ true -> PROXIMITY-DRIVEN. Pre-reg falsified; reframe topology-specificity.')
    else: print('VERDICT: distctrl between random and true -> PARTIAL. Report proximity carries some signal.')

## Cell 10 - Persistence check (did the runs reach Drive?)

In [ ]:
for s in SEEDS:
    dp=f'{DRIVE_RUNS}/topology_ablation/component0/L_upQdistctrl_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    print(f'  L_upQdistctrl_seed{s}: {os.path.isfile(dp)}')

## Done

3 new runs persist to Drive (L_upQdistctrl x seeds 11/13/17). Report the Cell 9 verdict + Cell 10 persistence back. If distctrl falls to the random level, the topology-specificity claim is hardened against the proximity objection and gets a new row in MECHANISM_MULTISEED.md + the paper's mechanism paragraph.